# Experiment 7: all-fish transition/static raster

This notebook follows the shared all-fish workflow used for experiments 1 and 5. It loads the experiment 7 cohort, validates its stimulus map, and generates z-score rasters, mean traces, left/right activity diagnostics, motion-period counts, and lifetime-sparseness figures.

Fish IDs, data location, blocks, and stimulus order come from `archive/notebooks/calcium/Exp7_trans_stat_several.ipynb`. The 25-second post-onset window follows its later configuration cell (the earlier cell used 32 seconds). The 8-second motion onset agrees with the experiment 7 stimulus configuration; the 6-second response extension follows the shared workflow.

Edit `USER_SETTINGS` and the figure controls beside each plot. With `data_base=None`, the shared loader discovers this computer's OneDrive Lab folders automatically. For a manual override, set `data_base` to the parent of the university OneDrive folder, such as `Path.home()` here or `Path(r'D:\Alejandro\Data')` on the other computer. All five fish must have the processed traces, significant-raster caches, and kept-neuron indices required by the shared loader. Reports are disabled until a run has been reviewed.


## 1. Setup and imports

In [ ]:
# Cell 01 - Setup
from pathlib import Path
import sys
import importlib
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_loading import resolve_lab_root

import src.plotting_common as plot_common
import src.several_fish_figures as several_figures
import src.several_fish_diagnostics as several_diagnostics
import src.several_fish_loading as several_loading
import src.several_fish_reporting as several_reporting
# Refresh the sorter before the plotting module that imports its function.
for module_name in ('src.neuron_ordering', 'src.plotting_all_fish', 'src.stimuli_timeline',
                    'src.response_metrics', 'src.several_fish_diagnostics'):
    importlib.reload(importlib.import_module(module_name))
for module in (plot_common, several_figures, several_loading, several_reporting):
    importlib.reload(module)


## Publication layout and export

Figures are drawn at their **final print size**, using DejaVu Sans: 8 pt axis labels and
titles, 7 pt ticks/legends, and 9 pt figure headings. DejaVu Sans ships with Matplotlib
and supports the scientific symbols here. Change `PUBLICATION_STYLE` below for a
journal-specific font. Re-run the plotting cells after changing the style.

An A4 sheet is 210 × 297 mm. With 15 mm margins the working area is 180 × 267 mm.
Small panels use about 85 mm width; dense grids use 180 mm width and 100–135 mm
height, leaving room for several panels, captions, and panel letters. Each 2 × 2
grid already contains four panels. Keep these dimensions when placing the figures;
shrinking a figure also shrinks its text. Sizes remain beside each plotting call.

Set `EXPORT_PUBLICATION_FIGURES=True` and run the export cell after reviewing the
plots. It writes editable PDF/SVG plus 600 dpi PNG into the configured folder.
The canvas is not cropped, so exported widths and heights match the millimetre
settings. These are individual figure blocks to assemble into a manuscript layout.


In [ ]:
# Physical sizes are millimetres; font sizes are points at the final print size.
MM = 1 / 25.4
PUBLICATION_STYLE = {
    'font.family': 'DejaVu Sans', 'font.size': 8,
    'axes.labelsize': 8, 'axes.titlesize': 8, 'figure.titlesize': 9,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'legend.title_fontsize': 7, 'axes.linewidth': 0.6,
    'lines.linewidth': 1.0, 'lines.markersize': 3,
    'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'figure.dpi': 150, 'savefig.dpi': 600, 'savefig.bbox': None,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'mathtext.fontset': 'dejavusans', 'figure.facecolor': 'white',
}
plt.rcParams.update(PUBLICATION_STYLE)
EXPORT_PUBLICATION_FIGURES = False  # Enable after reviewing the figures.
PUBLICATION_FORMATS = ('pdf', 'svg', 'png')
PUBLICATION_DPI = 600
PUBLICATION_DIR = repo_root / 'notebooks/calcium/exp_07_trans_stat/publication_figures/01'


## 2. Experiment 7 settings

In [ ]:
# Experiment 7 cohort and plotting settings from the archived several-fish notebook.
USER_SETTINGS = {
    'experiment_name': 'Exp_7_trans_stat',
    'analysis_label': 'Exp 7 transition/static',
    'data_base': None,  # Auto-discover OneDrive; use a parent Path for a manual override.
    'fish_ids': ['L742_f03', 'L742_f05', 'L742_f06', 'L742_f09', 'L742_f10'],
    'selected_blocks': ['B1', 'B2', 'B3', 'B4'],
    'verbose_loading': False,
    'timing': {
        'fps_2p': 2.0, 't_pre_s': 5.0, 't_post_s': 25.0,
        'motion_onset_s': 8.0, 'tau_s': 6.0,
    },
    # Preserve the archive's right-side group followed by its left-side group.
    'stim_order': [14, 9, 11, 13, 4, 6, 2, 7, 8, 10, 12, 3, 5, 1],
    'combine_mode': 'mean',  # 'mean' or 'concat'.
    'trace_type': 'zscore',
    'raster_sort_mode': 'left_right_index',  # Section 4 overrides this for its panel only.
    'left_control': 7,    # LeBcontrol
    'right_control': 14,  # RiBcontrol
}

# Check the archive's ID/name contract against the loaded experiment, not another experiment.
EXPECTED_STIMULUS_MAP = {
    'DF300Le': 1, 'DF300Ri': 2, 'F300Le': 3, 'F300Ri': 4,
    'F600Le': 5, 'F600Ri': 6, 'LeBcontrol': 7,
    'R300Le': 8, 'R300Ri': 9, 'R600Le': 10, 'R600Ri': 11,
    'RF600Le': 12, 'RF600Ri': 13, 'RiBcontrol': 14,
}

REPORT_SETTINGS = {
    'save_report': False,
    'run_label': 'exp7_trans_stat_all_fish_raster',
    'report_format': 'html',
    'export_notebook': True,
    'fallback_to_html': True,
    'comments': 'Enable only after reviewing the Exp 7 figures.',
}


## 3. Load and validate fish data

In [ ]:
# Load every configured fish and validate the experiment 7 stimulus mapping.
raster_inputs = several_loading.load_and_preflight_fish_raster_inputs(USER_SETTINGS)
globals().update(raster_inputs)
for fish_id, fish_data in all_fish_data.items():
    if any(fish_data['stimuli_id_map'].get(name) != stimulus_id
           for name, stimulus_id in EXPECTED_STIMULUS_MAP.items()):
        raise ValueError(f"Unexpected Exp 7 stimulus map for {fish_id}: {fish_data['stimuli_id_map']}")
stimuli_colors, stimuli_linestyles = plot_common.build_stimulus_style_maps(stimuli_names=stimuli_names)
print('Data root:', main_path)
print('Analysis root:', analysis_path)
print('Detected Exp 7 stimulus map:', stimuli_id_map)


## 4. All-fish z-score raster

Change `RASTER_SORT_MODE` below to `'corravg'` (Pearson distance with average
linkage), `'rastermap'`, `'left_right_index'` (the previous ordering), or
`'unsorted'`. Change `RASTER_SIDE` to `'both'`, `'left'`, or `'right'`, then
rerun the plotting cell. Left/right selects **stimulus conditions**, keeping
all neurons and the configured stimulus order. Corravg and Rastermap fit the
displayed conditions; selecting a side can therefore change the neuron order.
The left/right index always uses both control conditions.
After source-code updates, rerun section 1's setup cell before this cell;
it refreshes the sorter and plotting imports without reloading fish data.

Rastermap only supplies a row order: the original z-scores, mean trace and
panel styling are preserved. It is a display aid, not a test for neuron types.
If needed, run `%pip install "rastermap>=1.0,<2"` in this notebook's environment
and restart the kernel. See the [Rastermap project](https://github.com/MouseLand/rastermap).

This panel uses **89 × 65 mm** and **6 pt Arial** at final print size, so it fits
beside another panel on an A4 page. The local style applies to this raster only.
The four-column legend, mean trace, and colorbar are included inside that size.
Keep the exported dimensions when placing the panel in the manuscript.

[Nature's figure guide](https://research-figure-guide.nature.com/figures/building-and-exporting-figure-panels/)
specifies 89 mm single-column width and 5–7 pt text. Six points is the practical
small-text choice here; the figure-specific controls below remain editable.


In [ ]:
# Sorting and stimulus-side controls apply to this panel only.
RASTER_SORT_MODE = 'corravg'  # 'corravg', 'rastermap', 'left_right_index', 'unsorted'.
RASTER_SIDE = 'both'  # 'both', 'left', or 'right' stimulus conditions; all neurons stay.
RASTER_STIMULI_BY_SIDE = {
    'left': [7, 8, 10, 12, 3, 5, 1],
    'right': [14, 9, 11, 13, 4, 6, 2],
}
if RASTER_SIDE not in ('both', 'left', 'right'):
    raise ValueError("RASTER_SIDE must be 'both', 'left', or 'right'.")
raster_stim_order = [sid for sid in stim_order
                     if RASTER_SIDE == 'both' or sid in RASTER_STIMULI_BY_SIDE[RASTER_SIDE]]
raster_settings = {**settings, 'raster_sort_mode': RASTER_SORT_MODE}

# Final-size single-column panel; controls apply to this figure only.
RASTER_WIDTH_MM, RASTER_HEIGHT_MM = 89, 65
RASTER_FONT_PT = 6
RASTER_LEGEND_COLUMNS = 4
RASTER_STYLE = {
    **PUBLICATION_STYLE,
    'font.family': 'Arial', 'font.size': RASTER_FONT_PT,
    'axes.labelsize': RASTER_FONT_PT, 'axes.titlesize': RASTER_FONT_PT,
    'xtick.labelsize': RASTER_FONT_PT, 'ytick.labelsize': RASTER_FONT_PT,
    'legend.fontsize': RASTER_FONT_PT, 'legend.title_fontsize': RASTER_FONT_PT,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.6,
}
FIGSIZE = (RASTER_WIDTH_MM * MM, RASTER_HEIGHT_MM * MM)
with plt.rc_context(RASTER_STYLE):
    raster_result = several_figures.build_all_fish_raster_figure(
        all_fish_data, fish_ids, raster_stim_order, reference_fish, timing, raster_settings,
        stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
    )
    raster_figure, raster_axis = raster_result['figure'], raster_result['axes']
    raster_axis.set_title('All-fish z-score raster' +
                          ('' if RASTER_SIDE == 'both' else f' — {RASTER_SIDE} stimuli'))
    for axis in raster_figure.axes:
        plt.setp(axis.lines, linewidth=0.6)
    handles, labels = raster_axis.get_legend_handles_labels()
    raster_figure.legends[0].remove()
    raster_figure.legend(
        handles, labels, title='Movement onset', loc='lower center',
        ncol=RASTER_LEGEND_COLUMNS, frameon=False, handlelength=1.4,
        columnspacing=0.9, handletextpad=0.4, labelspacing=0.3,
    )
    raster_figure.subplots_adjust(left=0.13, right=0.87, bottom=0.40, top=0.91,
                                  hspace=0.10, wspace=0.14)
    plt.show()
flat_matrix_all_fish = raster_result['flat_matrix_all_fish']


## 4b. Independent stimulus panels: compare two layouts

Left and right get separate figures. Compare standalone rasters and mean traces
with the same traces **below their corresponding raster**. Each column is one
stimulus with its own relative time axis, averaging all four saved repetitions.
Every raster uses the same neuron identities and order. The original section 4
raster is unchanged.

Thin colored traces show individual fish; black is the equally weighted fish
mean, unlike the original pooled-neuron mean. Static periods are gray and
motion/flicker pale teal. Dotted lines mark appearance, solid teal lines the
active-period start, and dashed lines stimulus end, using Exp 07's loaded timing.
The phase legend sits in the upper-right corner; fish names are omitted.
Set `SAVE_PANEL_SVG=True` in the plotting cell to save all displayed layouts
to Fig. 2 `assets/generated`; `False` only displays them.

**Run the sorting cell below once.** Rerun it only when changing the sorting
method/reference or the loaded fish/stimulus data. Its prepared display stays in
memory. For appearance changes, edit and rerun **only the following plotting
cell**: side selection, font, dimensions, colors and limits never repeat sorting.
The default font is **6 pt**; `PANEL_FONT_PT` controls all text in these figures.
Dimensions are half the original width and height.


In [ ]:
# PREPARE / SORT — run once; rerun only for sorting or data changes.
PANEL_SORT_MODE = 'rastermap'  # 'corravg', 'rastermap', 'left_right_index', 'unsorted'.
PANEL_REUSE_RASTER_ORDER = False
panel_order = (raster_result['neuron_order'] if PANEL_REUSE_RASTER_ORDER
               and PANEL_SORT_MODE == raster_settings['raster_sort_mode'] else None)
stimulus_panel_display = several_diagnostics.build_stimulus_panel_display(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    {**settings, 'raster_sort_mode': PANEL_SORT_MODE}, neuron_order=panel_order,
)


In [ ]:
# APPEARANCE ONLY — rerun freely; this cell never computes the sorting.
PANEL_SIDE = 'both'  # 'both', 'left', or 'right'; each side gets its own figures.
PANEL_FONT_PT = 6  # Font-size control; normally keep at 6 pt or below.
PANEL_RASTER_FIGSIZE = (150 * MM, 45 * MM)
PANEL_MEAN_FIGSIZE = (150 * MM, 42.5 * MM)
PANEL_COMBINED_FIGSIZE = (150 * MM, 65 * MM)
PANEL_Z_LIMITS = raster_result['image'].get_clim()
PANEL_MEAN_YLIM = None  # One automatic range shared by all fish/conditions.
PANEL_CMAP = 'Greys'
SAVE_PANEL_SVG = False  # True saves every displayed side/layout as SVG.
PANEL_SVG_DIR = resolve_lab_root() / (
    'Results/Paper/figures/main/fig02/assets/generated'
)
PANEL_STYLE = {
    **PUBLICATION_STYLE, 'font.family': 'Arial', 'font.size': PANEL_FONT_PT,
    'axes.labelsize': PANEL_FONT_PT, 'axes.titlesize': PANEL_FONT_PT,
    'figure.titlesize': PANEL_FONT_PT, 'xtick.labelsize': PANEL_FONT_PT,
    'ytick.labelsize': PANEL_FONT_PT, 'legend.fontsize': PANEL_FONT_PT,
    'legend.title_fontsize': PANEL_FONT_PT,
}
if PANEL_SIDE not in ('both', 'left', 'right'):
    raise ValueError("PANEL_SIDE must be 'both', 'left', or 'right'.")
panel_sides = {side: [sid for sid in stimulus_panel_display['stimulus_ids']
                      if sid in RASTER_STIMULI_BY_SIDE[side]]
               for side in ('left', 'right') if PANEL_SIDE in ('both', side)}
with plt.rc_context(PANEL_STYLE):
    stimulus_panel_result = several_figures.plot_all_fish_stimulus_figures(
        stimulus_panel_display, panel_sides,
        z_limits=PANEL_Z_LIMITS, mean_ylim=PANEL_MEAN_YLIM,
        raster_figsize=PANEL_RASTER_FIGSIZE, mean_figsize=PANEL_MEAN_FIGSIZE,
        combined_figsize=PANEL_COMBINED_FIGSIZE, cmap=PANEL_CMAP,
    )
    for panel_name, figure in stimulus_panel_result['figures'].items():
        if SAVE_PANEL_SVG:
            plot_common.save_publication_figure(
                figure, PANEL_SVG_DIR / f'exp07_stimulus_panels_{panel_name}',
                formats=('svg',), dpi=PUBLICATION_DPI,
            )
        display(figure)
        plt.close(figure)


## 5. Mean z-score traces

In [ ]:
# Mean z-score traces in the same experiment 7 stimulus order.
FIGSIZE = (180 * MM, 65 * MM)
mean_trace_result = several_figures.build_plot_all_fish_mean_zscore_traces(
    all_fish_data, fish_ids, stim_order, reference_fish, timing,
    stimuli_colors, stimuli_linestyles, figsize=FIGSIZE,
)
plt.show()


## 6. Left/right activity diagnostics

In [ ]:
# One activity definition for section 6 and the shared-neuron analysis below.
SECTION6_ACTIVE_SETTINGS = {
    'active_fraction_threshold': 0.10, 'min_epoch_s': 1.0,
    'min_active_reps': 2, 'expected_reps': 4, 'require_expected_reps': True,
}
# Matched left/right groups from the archive's stimulus map.
OVERLAP_FIGSIZE = (85 * MM, 80 * MM)
RASTER_FIGSIZE = (180 * MM, 100 * MM)
OVERLAP_STIMULI = {
    'left': [7, 8, 10, 12, 3, 5, 1],
    'right': [14, 9, 11, 13, 4, 6, 2],
}
overlap_results = several_figures.plot_left_right_active_overlap_diagnostics(
    all_fish_data, fish_ids, reference_fish, timing, OVERLAP_STIMULI,
    motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
    show_overlap=True, show_significant_raster=True,
    **SECTION6_ACTIVE_SETTINGS,
    overlap_figsize=OVERLAP_FIGSIZE, raster_figsize=RASTER_FIGSIZE,
)


## 6b. Response strength in neurons shared with the control

For each fish and side, select **active to control AND active to comparison
stimulus**, separately for each pair. Reuse the exact binary decisions returned
by section 6, using `SECTION6_ACTIVE_SETTINGS` above. Rerun section 6 after changing
those settings. No neuron is classified again from its continuous z-score.

For each shared neuron, integrate its signed z-score from that condition's loaded
**motion onset to motion end + 5 s**. Initial static activity is excluded; the
extra 5 s captures the decay. Average AUC across fully finite repetitions, using
all trials rather than only active trials. Keep the original normalization and
signed responses, with no additional baseline subtraction or clipping at zero.

Calculate **control minus comparison** within the same neurons, then average
paired differences per fish. Positive values indicate stronger control responses.
The default plot uses **duration-normalized AUC**, dividing each AUC by its actual
sample integration span; choose `raw_auc` for integrated z-score seconds instead.
Both measurements are retained because condition-specific motion lengths can
differ. Actual bounds and any clipping are recorded in `shared_auc_windows`.

Gray points show fish; black averages fish equally. Empty intersections and
missing pairs remain undefined, with shared/finite-neuron counts retained in the
tables. The summary below shows the range of shared-neuron counts across fish.
This is a descriptive comparison within the observed-trial selected population.


In [ ]:
# Calculate once from section 6; change only the plot cell for appearance edits.
SHARED_AUC_DECAY_S = 5.0
SHARED_AUC_CONTROLS = {'left': settings['left_control'], 'right': settings['right_control']}
shared_auc_result = several_diagnostics.build_shared_control_auc_analysis(
    all_fish_data, fish_ids, overlap_results, SHARED_AUC_CONTROLS, timing,
    decay_s=SHARED_AUC_DECAY_S,
)
display(shared_auc_result['tables']['shared_auc_summary'][[
    'side', 'condition', 'n_fish', 'n_shared_min', 'n_shared_max',
    'mean_delta_normalized_auc',
]])


In [ ]:
# Appearance only. Positive = control stronger; negative = comparison stronger.
SHARED_AUC_METRIC = 'normalized_auc'  # 'normalized_auc' or 'raw_auc'.
SHARED_AUC_FONT_PT = 6
SHARED_AUC_FIGSIZE = (85 * MM, 60 * MM)  # Per side.
SHARED_AUC_STYLE = {
    **PUBLICATION_STYLE, 'font.size': SHARED_AUC_FONT_PT,
    'axes.labelsize': SHARED_AUC_FONT_PT, 'axes.titlesize': SHARED_AUC_FONT_PT,
    'xtick.labelsize': SHARED_AUC_FONT_PT, 'ytick.labelsize': SHARED_AUC_FONT_PT,
    'legend.fontsize': SHARED_AUC_FONT_PT,
}
with plt.rc_context(SHARED_AUC_STYLE):
    shared_auc_figures = several_figures.plot_shared_control_auc(
        shared_auc_result, metric=SHARED_AUC_METRIC, figsize=SHARED_AUC_FIGSIZE,
    )
    for figure, axis in shared_auc_figures.values():
        display(figure)
        plt.close(figure)


## 6c. Combined population and shared-neuron summary

One three-row figure per side: population activity, recruitment, and response
strength in pairwise shared neurons. Stimulus labels appear only at the bottom;
one legend identifies sage fish dots and the slate-blue equal-fish mean.
The last row leaves Control as n/a because no self-comparison is measured.

Preparation reuses notebook 02's canonical recruitment/activity workflow with
section 6's activity settings. Original windows remain distinct: population
activity uses the observed motion-plus-12 s window; shared-neuron AUC uses motion
plus 5 s. These rows summarize different populations and measurements.


In [ ]:
# Prepare once after section 6b; appearance changes use only the next cell.
SUMMARY_SIDE_GROUPS = {
    'left': {'control': 'LeBcontrol',
             'rocking': ['R300Le', 'R600Le', 'RF600Le'],
             'flicker': ['F300Le', 'F600Le', 'DF300Le']},
    'right': {'control': 'RiBcontrol',
              'rocking': ['R300Ri', 'R600Ri', 'RF600Ri'],
              'flicker': ['F300Ri', 'F600Ri', 'DF300Ri']},
}
SUMMARY_CONDITION_LABELS = ['Control', 'R300', 'R600', 'RF600', 'F300', 'F600', 'DF300']
combined_summary_analysis = several_diagnostics.build_recruitment_containment_analysis(
    all_fish_data, fish_ids, SUMMARY_SIDE_GROUPS, timing,
    SECTION6_ACTIVE_SETTINGS, SUMMARY_CONDITION_LABELS,
)


Change dimensions, font, and colors below without rebuilding the analysis.
Set `SAVE_SUMMARY_SVG=True` to export both combined figures to Fig. 2
`assets/generated`; `False` only displays them.


In [ ]:
# Appearance and optional SVG export only.
SUMMARY_FIGSIZE = (85 * MM, 110 * MM)  # Per side: width, height.
SUMMARY_FONT_PT = 6
SUMMARY_FISH_COLOR = '#9AAF9F'  # Muted sage.
SUMMARY_MEAN_COLOR = '#385F73'  # Slate blue.
SAVE_SUMMARY_SVG = False
SUMMARY_SVG_DIR = resolve_lab_root() / 'Results/Paper/figures/main/fig02/assets/generated'
SUMMARY_STYLE = {**PUBLICATION_STYLE, 'font.family': 'Arial',
    **{key: SUMMARY_FONT_PT for key in ('font.size', 'axes.labelsize', 'axes.titlesize',
       'figure.titlesize', 'xtick.labelsize', 'ytick.labelsize', 'legend.fontsize')}}
with plt.rc_context(SUMMARY_STYLE):
    combined_summary_figures = several_figures.plot_recruitment_strength_summary(
        combined_summary_analysis, shared_auc_result, figsize=SUMMARY_FIGSIZE,
        fish_color=SUMMARY_FISH_COLOR, mean_color=SUMMARY_MEAN_COLOR,
    )
    for side, (figure, axes) in combined_summary_figures.items():
        if SAVE_SUMMARY_SVG:
            plot_common.save_publication_figure(
                figure, SUMMARY_SVG_DIR / f'exp07_combined_recruitment_strength_{side}',
                formats=('svg',), dpi=PUBLICATION_DPI,
            )
        display(figure)
        plt.close(figure)


## 7. Motion-period active-neuron counts

In [ ]:
# Motion-period active-neuron counts, preserving the configured stimulus order.
FIGSIZE = (180 * MM, 65 * MM)
motion_active_result = several_figures.plot_motion_active_neuron_counts(
    all_fish_data, fish_ids, reference_fish, timing, stim_order, stimuli_colors,
    figsize=FIGSIZE, motion_onset_s=timing['motion_onset_s'], tau_s=timing['tau_s'],
)
active_neuron_counts = motion_active_result['counts']


## 8. Lifetime sparseness and high-sparseness raster

In [ ]:
# Lifetime sparseness across the selected experiment 7 stimuli.
SPARSENESS_FIGSIZE = (85 * MM, 75 * MM)
RASTER_FIGSIZE = (180 * MM, 100 * MM)
SPARSENESS_STIMULI = stim_order  # Edit to select a subset of IDs or names.
HIGH_LIFETIME_SPARSENESS = 0.75
HIGH_SPARSENESS_COMBINE_MODE = 'mean'
sparseness_result = several_figures.plot_lifetime_sparseness_analysis(
    all_fish_data, fish_ids, reference_fish, timing, SPARSENESS_STIMULI,
    HIGH_LIFETIME_SPARSENESS, stimuli_colors, stimuli_linestyles,
    combine_mode=HIGH_SPARSENESS_COMBINE_MODE,
    sparseness_figsize=SPARSENESS_FIGSIZE, raster_figsize=RASTER_FIGSIZE,
    analysis_label=settings['analysis_label'],
)


## 9. Optional analysis report

In [ ]:
# Cell 09 - Optional reviewed-run report
# Save this notebook before enabling REPORT_SETTINGS['save_report'] so the HTML includes current outputs.
notebook_path = repo_root / 'notebooks' / 'calcium' / 'exp_07_trans_stat' / '01_all_fish_raster.ipynb'
report_summary = pd.DataFrame([
    {
        'fish_id': fish_id,
        'kept_neurons': len(fish_data['kept_neuron_indices']),
        'detected_stimuli': len(fish_data['stimuli_ids']),
    }
    for fish_id, fish_data in all_fish_data.items()
])
report_result = several_reporting.save_analysis_report_run(
    settings=settings,
    report_settings=REPORT_SETTINGS,
    analysis_path=analysis_path,
    tables={'run_summary': report_summary, **shared_auc_result['tables']},
    notebook_path=notebook_path,
    report_name='01_all_fish_raster_report',
    extra_metadata={
        'stim_order': stim_order,
        'combine_mode': settings['combine_mode'],
        'trace_type': settings['trace_type'],
        'raster_sort_mode': raster_settings['raster_sort_mode'],
        'raster_side': RASTER_SIDE,
        'raster_stim_order': raster_stim_order,
        'all_fish_matrix_shape': flat_matrix_all_fish.shape,
        'section6_activity_settings': SECTION6_ACTIVE_SETTINGS,
        'shared_auc_decay_s': SHARED_AUC_DECAY_S,
        'shared_auc_controls': SHARED_AUC_CONTROLS,
        'shared_auc_plot_metric': SHARED_AUC_METRIC,
        'shared_auc_difference': 'control_minus_stimulus',
    },
)
report_result


## Export reviewed publication figures

Run after the plotting cells; the style cell controls whether files are written.


In [ ]:
publication_figures = {
    'all_fish_raster': raster_result['figure'],
    'mean_zscore_traces': mean_trace_result['figure'],
    'motion_active_counts': motion_active_result['figure'],
    'lifetime_sparseness': sparseness_result['sparseness_figure'],
    'high_sparseness_raster': sparseness_result['high_sparseness_figure'],
}
publication_figures.update({f'stimulus_panels_{name}': figure
                            for name, figure in stimulus_panel_result['figures'].items()})
publication_figures.update({f'shared_control_auc_{side}': figure
                            for side, (figure, axis) in shared_auc_figures.items()})
publication_figures.update({f'combined_recruitment_strength_{side}': figure
                            for side, (figure, axes) in combined_summary_figures.items()})
for side, result in overlap_results.items():
    for key in ('overlap_figure', 'diagnostic_figure'):
        if key in result:
            publication_figures[f'{side}_{key}'] = result[key]
if EXPORT_PUBLICATION_FIGURES:
    for name, figure in publication_figures.items():
        if figure is not None:
            plot_common.save_publication_figure(
                figure, PUBLICATION_DIR / name,
                formats=PUBLICATION_FORMATS, dpi=PUBLICATION_DPI,
            )
    print('Publication figures:', PUBLICATION_DIR)
